In [4]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# Sample input DataFrame
data = pd.DataFrame({
    "text": [
        "The robot gazed at the stars, wondering about its creator.",
        "In the shimmering city of tomorrow, all was not well.",
        "Humanity's last hope was hidden in the ruins of the old world."
    ],
    "model": ["gpt-4", "llama-2", "gemini"],
    "temperature": [0.7, 1.0, 0.9],
    "batch_id": ["123", "123", "123"]
})

class LabellingBatch:
    text_area = widgets.Textarea(value='', description='Text:', layout=widgets.Layout(width="100%", height="500px"))
    usage_text = widgets.Combobox(value='', description='Usage:', layout=widgets.Layout(width="100%", height="20px"), ensure_option=False, options=['dialogue', 'exposition', 'opener', 'style'])
    model_label = widgets.Label(value='', layout=widgets.Layout(width="100%"))
    temperature_label = widgets.Label(value='', layout=widgets.Layout(width="100%"))
    batch_id_label = widgets.Label(value='', layout=widgets.Layout(width="100%"))
    rating = widgets.RadioButtons(
            options=["bad", "ok", "amazing"],
            value="bad",
            description='Rating:',
            layout=widgets.Layout(width="50%")
        )
    next_button = widgets.Button(description="Next", button_style='success')
    save_button = widgets.Button(description="Save labels", button_style='success')
    output = widgets.Output()

    input_df: pd.DataFrame
    outputs = []
    current_index: int
    
    def label_data(self, input_df: pd.DataFrame, experiment_name, current_index: int = 0):

        self.output_df: pd.DataFrame = pd.DataFrame()
        self.current_index = current_index
        self.input_df = input_df
        self.experiment_name = experiment_name
        
        # Attach event listener
        self.next_button.on_click(self.submit_and_next)
        self.save_button.on_click(self.write_labels)
        
        # Display initial data
        self.update_widgets(current_index)
        
        # Layout the widgets
        display(widgets.VBox([
            self.model_label,
            self.temperature_label,
            self.batch_id_label,
            self.text_area,
            self.usage_text,
            self.rating,
            self.next_button,
            self.save_button,
            self.output
        ]))

    def update_widgets(self, index: int):
        """Update widgets with the current row data."""
        df = self.input_df
        self.text_area.value = df.loc[index, "text"]
        self.usage_text.value = ""
        self.model_label.value = f"Model: {df.loc[index, 'model']}"
        self.temperature_label.value = f"Temperature: {df.loc[index, 'temperature']}"
        self.batch_id_label.value = f"Project: {df.loc[index, 'project_name']} Experiment: {df.loc[index, 'experiment_name']} Batch id: {df.loc[index, 'batch_id']}"
        self.rating.value = "bad"
    
    def submit_and_next(self, _):
        """Save current values and move to the next row."""
        new_row = {
            "target_text": self.text_area.value,
            "label": self.rating.value,
            "usage_text": self.usage_text.value,
            "input_index": self.current_index,
        }

        for column in ["text", "temperature", "model", "batch_id"]:
            new_row[column] = self.input_df.loc[self.current_index, column]

        self.outputs.append(new_row)
        
        # Increment index
        self.current_index += 1
        
        # Check if we reached the end
        if self.current_index < len(self.input_df):
            self.update_widgets(self.current_index)
        else:
            with output:
                clear_output()
                print("All entries have been labeled!")
        
        # Clear previous messages
        with output:
            clear_output()
            print(f"Entry {self.current_index}/{len(self.input_df)} labeled.")

    def write_labels(self, _):
        pd.DataFrame(self.outputs).to_parquet(f"labels/generate_writing/{self.experiment_name}.parquet")




In [5]:
import os

PROJECT_NAME = "QoGD"
EXPERIMENT_NAME = "act2fill1"

folder = f"generated_text/{PROJECT_NAME}/{EXPERIMENT_NAME}/"
sorted(os.listdir(folder))


['.ipynb_checkpoints',
 '2025-02-02 10:06:21.895171',
 '2025-02-02 10:36:13.224987',
 '2025-02-02 10:38:05.308279',
 '2025-02-02 10:38:59.375849',
 '2025-02-02 10:40:59.200546',
 '2025-02-02 10:42:32.483271',
 '2025-02-02 10:43:39.922796',
 '2025-02-02 10:45:12.310464',
 '2025-02-02 10:52:51.196961',
 '2025-02-02 10:53:39.074500',
 '2025-02-02 10:55:38.979072',
 '2025-02-02 10:58:05.646643',
 '2025-02-02 10:59:30.719864',
 '2025-02-02 11:01:49.950718',
 '2025-02-02 11:03:59.768551',
 '2025-02-02 11:06:19.881430',
 '2025-02-02 11:08:03.867395',
 '2025-02-02 11:11:26.789073',
 '2025-02-02 11:13:29.421156',
 '2025-02-02 11:15:55.545324',
 '2025-02-02 11:17:50.316457',
 '2025-02-02 11:20:49.329630',
 '2025-02-02 11:22:30.402177',
 '2025-02-02 11:25:31.749131',
 '2025-02-02 22:45:46.296854',
 '2025-02-03 18:52:29.401501',
 '2025-02-03 18:56:05.232001',
 '2025-02-03 18:58:35.962751',
 '2025-02-03 19:11:42.352805',
 '2025-02-03 19:14:17.389578',
 '2025-02-03 19:16:12.067358',
 '2025-02-03 19:

In [6]:
index = 10
with open(f"{folder}/{sorted(os.listdir(folder))[index]}/user_prompt.txt", 'r') as f:
    print(f.read()[:200])

Here is the outline and text so far. Write part 2. Don't return what's already written.

### Beliefs explored

Part 1:
1. There is no simulation - we already exited it. We're outside of Plato's cave. 


In [7]:
ts = sorted(os.listdir(folder))[index]
input_df = pd.read_parquet(f"{folder}{ts}/dataframe.parquet")
labeller = LabellingBatch()
labeller.label_data(input_df, PROJECT_NAME + "-" + EXPERIMENT_NAME + "-" + ts)

In [5]:
len(labeller.outputs)


10

In [7]:
df = pd.read_parquet(f"labels/generate_writing/first_attempt.parquet")
df.size

80